# 📘 실전 프로젝트

지금까지 배운 OpenCV 기술을 종합적으로 활용하는 프로젝트입니다.

**학습 목표:**
- 이미지 파노라마(연결) 기법
- 워터마크 삽입
- 이미지 품질 측정 (PSNR)
- 실전 이미지 처리 파이프라인

## 1. 이미지에 워터마크 삽입

워터마크는 이미지 저작권 보호를 위해 텍스트나 로고를 겹치는 기법입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  워터마크 삽입                           │
# │  텍스트 워터마크 + 투명도 조절            │
# └─────────────────────────────────────────┘

# 원본 이미지 생성 (풍경 느낌)
img = np.zeros((400, 600, 3), dtype=np.uint8)
# 하늘
for y in range(200):
    img[y, :] = [255-y//2, 180-y//3, 50+y//4]  # BGR
# 땅
img[200:, :] = [40, 120, 40]

# 태양
cv2.circle(img, (480, 80), 40, (0, 200, 255), -1)
# 산
pts = np.array([[100, 200], [250, 100], [400, 200]], np.int32)
cv2.fillPoly(img, [pts], (60, 100, 60))

# 워터마크 생성 (투명도 포함)
overlay = np.zeros_like(img)
cv2.putText(overlay, 'WATERMARK', (100, 220),
            cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 4)
cv2.putText(overlay, 'OpenCV', (200, 280),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (200, 200, 200), 2)

# 투명도 적용 (알파 블렌딩)
alpha = 0.3  # 워터마크 투명도
img_watermarked = cv2.addWeighted(img, 1, overlay, alpha, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('원본')
axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[1].set_title('워터마크')
axes[2].imshow(cv2.cvtColor(img_watermarked, cv2.COLOR_BGR2RGB)); axes[2].set_title(f'결과 (alpha={alpha})')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("💡 cv2.addWeighted(): 두 이미지를 가중합으로 섞음")
print(f"   결과 = img×{1-alpha} + overlay×{alpha}")

## 2. 이미지 품질 측정 (PSNR)

**PSNR**(Peak Signal-to-Noise Ratio)은 이미지 품질을 측정하는 지표입니다.
원본과 처리된 이미지 간의 차이를 정량화합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  이미지 품질 측정 (PSNR)                │
# │  노이즈 제거 효과를 정량적으로 비교      │
# └─────────────────────────────────────────┘

# 깨끗한 이미지 생성
img_clean = np.zeros((200, 300, 3), dtype=np.uint8)
img_clean[:] = (180, 200, 220)
cv2.circle(img_clean, (150, 100), 60, (0, 150, 255), -1)
cv2.rectangle(img_clean, (50, 50), (100, 150), (255, 100, 100), -1)

# 다양한 노이즈 수준 추가
rng = np.random.default_rng(42)

def add_gaussian_noise(img, sigma):
    noise = rng.normal(0, sigma, img.shape).astype(np.int16)
    noisy = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return noisy

def psnr(img1, img2):
    mse = np.mean((img1.astype(float) - img2.astype(float)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(255.0 ** 2 / mse)

# 노이즈 이미지와 복원 비교
img_noisy = add_gaussian_noise(img_clean, 30)
img_denoised = cv2.GaussianBlur(img_noisy, (9, 9), 0)
img_median = cv2.medianBlur(img_noisy, 5)

psnr_noisy = psnr(img_clean, img_noisy)
psnr_denoised = psnr(img_clean, img_denoised)
psnr_median = psnr(img_clean, img_median)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
images = [img_clean, img_noisy, img_denoised, img_median]
titles = [f'원본', f'노이즈 (PSNR: {psnr_noisy:.1f}dB)',
          f'가우시안 블러 (PSNR: {psnr_denoised:.1f}dB)',
          f'미디안 블러 (PSNR: {psnr_median:.1f}dB)']
for ax, im, t in zip(axes, images, titles):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

print("💡 PSNR이 높을수록 원본에 가까운 고품질 이미지")
print("💡 가우시안 블러: 일반적인 노이즈 제거에 효과적")
print("💡 미디안 블러: 소금/후추 노이즈에 특히 효과적")

## 3. 종합 — 이미지 처리 파이프라인

실전에서 자주 사용하는 이미지 처리 파이프라인을 구성합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  종합 이미지 처리 파이프라인              │
# │  읽기 → 전처리 → 분석 → 결과 출력        │
# └─────────────────────────────────────────┘

# 1. 원본 이미지 생성
img = np.zeros((400, 600, 3), dtype=np.uint8)
img[:] = (220, 220, 220)  # 밝은 배경

# 여러 도형 그리기
cv2.rectangle(img, (50, 50), (200, 200), (0, 100, 255), -1)   # 주황 사각형
cv2.circle(img, (350, 150), 70, (255, 0, 0), -1)               # 파란 원
cv2.circle(img, (500, 300), 50, (0, 255, 0), -1)               # 초록 원
cv2.rectangle(img, (100, 250), (250, 370), (0, 0, 255), -1)   # 빨간 사각형

# 노이즈 추가
noisy = img.copy()
for _ in range(500):
    x, y = rng.integers(0, 600), rng.integers(0, 400)
    noisy[y, x] = [0, 0, 0] if rng.random() > 0.5 else [255, 255, 255]

# 2. 전처리 파이프라인
gray = cv2.cvtColor(noisy, cv2.COLOR_BGR2GRAY)
denoised = cv2.medianBlur(gray, 5)
_, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel = np.ones((3, 3), np.uint8)
cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)

# 3. 분석 (윤곽선 검출)
contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 4. 결과 출력
result = noisy.copy()
areas = []
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area > 500:  # 작은 노이즈 제외
        areas.append(area)
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(result, f'{area:.0f}', (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
steps = [
    (cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB), '1. 노이즈 원본'),
    (gray, '2. 흑백 변환'),
    (denoised, '3. 미디안 블러'),
    (binary, '4. Otsu 이진화'),
    (cleaned, '5. 모폴로지 닫기'),
    (cv2.cvtColor(result, cv2.COLOR_BGR2RGB), f'6. 결과 ({len(areas)}개 객체)')
]
for ax, (im, t) in zip(axes.flat, steps):
    if len(im.shape) == 2:
        ax.imshow(im, cmap='gray')
    else:
        ax.imshow(im)
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"검출된 객체 수: {len(areas)}")
print(f"면적: {areas}")
print(f"\n=== 파이프라인 요약 ===")
print("1. 흑백 변환 → 2. 노이즈 제거 → 3. 이진화 → 4. 모폴로지 → 5. 윤곽선 검출")

## 🎯 연습 문제

1. 다양한 alpha 값(0.1, 0.3, 0.5, 0.7)으로 워터마크를 삽입하고 결과를 비교하세요.
2. PSNR을 사용하여 가우시안 블러의 커널 크기별 노이즈 제거 효과를 비교하세요.
3. 이미지 처리 파이프라인에 엣지 검출(Canny)을 추가하고, 윤곽선 대신 엣지를 시각화하세요.
4. 히스토그램 평활화(`cv2.equalizeHist()`)를 적용하여 대비를 개선하는 코드를 작성하세요.
5. 이미지를 4등분하여 각 영역의 평균 밝기를 계산하고 밝은/어두운 영역을 표시하세요.